## GPT prompting: n80 10k examples, third test

### requires python >= 3.10

### runs through all three prompts and saves each result into individual file

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *

In [3]:
from prompts.semantic_categories.v02.prompt import ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS, ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS, TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
RESULTS_DIR = "../../results/"

EXAMPLE_FILE = "n80_examples_large_v01/gpt_v02/" + "gpt_10K_b10_run01.csv"

GPT_ANSWER_FILE_ALIVE = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_alive.csv"
GPT_ANSWER_FILE_ABSTRACT = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_abstract.csv"
GPT_ANSWER_FILE_TIME = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_timex.csv"

GPT_FILTERED_FILE = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_filtered.csv"

CONF_FILE = 'azure.ini'

# OSA I : Andmed


## testimise põhjusel on kasutusel vana 10k v1 andmefail, et tulemusi saaks võrrelda

In [4]:
df = pd.read_csv(RESULTS_DIR+EXAMPLE_FILE, encoding="utf-8",  sep=",")

In [6]:
# võtta need, mille puhul gpt ütles "no"

spatial_obl_ex = df[df["classification2"]=="no"]

# OSA II : GPT

## GPT jaoks vajalik

In [7]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [14]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

# ALIVE

## Andmete söötmine

In [ ]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": in2json(user_payload)}
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [17]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [ ]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunks(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ALIVE_FEW_SHOTS_STR, ALIVE_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= 151000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

In [32]:
#used_tokens # batch 10-> 3700,  10K lauset -> ~ 7.9 eur

3726

In [21]:
#len(results)

10000

## andmed tabelisse 


In [22]:
if len(results) == len(df):
    df["is_alive"] = [r["a"] for r in results]

In [ ]:
saving_fname = RESULTS_DIR+GPT_ANSWER_FILE_ALIVE 
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# ABSTRACT

In [ ]:
df = spatial_obl_ex

In [ ]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunks(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= 351000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

In [ ]:
if len(results) == len(df):
    df["is_abstract"] = [r["a"] for r in results]

In [ ]:
saving_fname = RESULTS_DIR+GPT_ANSWER_FILE_ABSTRACT 
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# TIMEX

In [ ]:
df = spatial_obl_ex

In [ ]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunks(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, TIME_FEW_SHOTS_STR, TIME_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= 151000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

In [ ]:
if len(results) == len(df):
    df["is_time"] = [r["a"] for r in results]

In [ ]:
saving_fname = RESULTS_DIR+GPT_ANSWER_FILE_TIME
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# Kokku filtreeritud fail

In [ ]:
fname1 = RESULTS_DIR+GPT_ANSWER_FILE_ALIVE
fname2 = RESULTS_DIR+GPT_ANSWER_FILE_ABSTRACT
fname3 = RESULTS_DIR+GPT_ANSWER_FILE_TIME

df1 = pd.read_csv(fname1, encoding="utf-8",  sep=",")
df2 = pd.read_csv(fname2, encoding="utf-8",  sep=",")
df3 = pd.read_csv(fname3, encoding="utf-8",  sep=",")

In [ ]:
df2_selected = df2[['head_id', "verb", "verb_compound", "morph_case", "form", "is_abstract"]]
df3_selected = df3[['head_id', "verb", "verb_compound", "morph_case", "form", "is_time"]]

# Merge df1 with df2_selected on 'id'
merged_df = df1.merge(df2_selected, on=['head_id', "verb", "verb_compound", "morph_case", "form"], how='left')

# Merge the result with df3_selected on 'id'
final_df = merged_df.merge(df3_selected, on=['head_id', "verb", "verb_compound", "morph_case", "form"], how='left')

In [ ]:
final_df.to_csv(RESULTS_DIR+GPT_FILTERED_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)